# 1. Datos sintéticos etiquetados por construcción

Laya aprende de ejemplos, y escribir a mano miles de tickets con su categoría, prioridad y bloqueo no escala.
`generar.py` le pide a un LLM que escriba tickets **para una respuesta decidida de antemano**: cada llamada es para
una combinación fija de categoría × prioridad × bloqueo, así que la etiqueta se conoce sin revisar el texto.

| | |
|---|---|
| Combinaciones | 6 categorías × 4 prioridades × bloqueo (baja y media nunca bloquean) = **36** |
| Contexto | `--prompt` o un archivo con uno por línea (`data/contextos.txt`, 48 sectores y países) |
| Señales | cada categoría exige en los hechos sus señales propias (un correo que pide la contraseña, una alerta del antivirus…), sin nombrarla |
| Validación | Pydantic genera el esquema que restringe la salida y rechaza lotes en texto libre, campos vacíos o descripciones de menos de 25 palabras |
| Filtros | frases que delatan la respuesta, el nombre de la propia categoría, títulos repetidos o copiados de los 20 tickets de prueba |

Backends medidos en esta tarea:

| Backend | Modelo | Velocidad | Costo |
|---|---|---|---|
| Ollama (local) | `gemma3:12b` en una RTX 4070 Ti SUPER | ~12 tickets por minuto | gratis |
| OpenRouter | `openai/gpt-5.6-luna` | 216 tickets en ~44 s con 18 hilos | ~US$0,30 por 1.000 |
| OpenAI | `gpt-6-luna` | ~5 s por llamada | respaldo automático si OpenRouter se queda sin saldo |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import modelo  # Antes que torch: caché de Hugging Face en el proyecto y llaves locales.

import random

import generar

print(len(generar.SPECS), "combinaciones; ejemplo:", generar.SPECS[0])
print(generar.prompt_for(("seguridad", "critica", True), "tickets de un banco en España", [], random.Random(0)))

## Generar

La forma cómoda es la línea de comandos; las filas se **añaden** a `data/sintetico.csv` (ignorado en git) en cuanto
termina cada combinación, así que cortar la corrida no pierde lo generado. Esta celda genera 36 casos de ejemplo con
OpenRouter (unos 2 centavos); cambia `--backend ollama` para hacerlo gratis en local.

In [ ]:
!cd {ROOT} && {sys.executable} generar.py --backend openrouter --n 36 --prompt "tickets de una clínica en Uruguay" --salida data/ejemplo.csv

# El dataset completo del README se generó así (~10.000 casos, ~35 min, US$3):
# python generar.py --backend openrouter --contextos data/contextos.txt --n 216 --hilos 18

## Cómo quedó el dataset

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

rows = generar.read()
print(len(rows), "casos en", generar.CSV_PATH.relative_to(ROOT))
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, (title, values) in zip(axes, [("Categoría", [r["referencia"][0] for r in rows]),
                                      ("Prioridad", [r["referencia"][1] for r in rows]),
                                      ("Bloquea", [str(r["bloquea"]) for r in rows])]):
    counts = Counter(values)
    ax.bar(list(counts), list(counts.values()), color="#4fa8f0", edgecolor="#0d0d0d")
    ax.set_title(title)
plt.tight_layout()

In [ ]:
words = [len(r["descripcion"].split()) for r in rows]
plt.figure(figsize=(8, 3))
plt.hist(words, bins=40, color="#2fbf94", edgecolor="#0d0d0d")
plt.title("Palabras por descripción")
plt.xlabel("palabras")
contexts = Counter(r["contexto"] for r in rows)
print(len(contexts), "contextos distintos; los más frecuentes:")
for context, n in contexts.most_common(5):
    print(f"  {n:5}  {context[:90]}")

In [ ]:
for r in random.Random(0).sample(rows, 3):
    print(f"[{r['referencia'][0]} · {r['referencia'][1]} · bloquea={r['bloquea']}] {r['titulo']}\n{r['descripcion']}\n")